# ADVC — Colab Setup & Phase 1 Evaluation
Run cells **top to bottom** on a fresh Colab session.

> **Before starting:** Make sure GPU is enabled — Runtime → Change runtime type → T4 GPU

## 1 — Check GPU

In [ ]:
!nvidia-smi

## 2 — Clone repo & navigate

In [ ]:
%cd /content
!git clone https://github.com/Jmanav/ADVC.git ADVC-main
%cd ADVC-main

## 3 — Install dependencies

In [ ]:
!pip install -r requirements.txt

## 4 — Verify PyTorch + GPU

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('Torch version  :', torch.__version__)

## 5 — Kaggle setup
**Before running this cell:**
1. Go to `https://www.kaggle.com` → profile → Settings → API → **Create New Token** — downloads `kaggle.json`
2. Accept competition terms at `https://www.kaggle.com/competitions/imagenet-object-localization-challenge` → **Join Competition**

In [ ]:
# Uploads kaggle.json via file picker
from google.colab import files
import os, pathlib

uploaded = files.upload()  # select kaggle.json from your machine

kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
os.rename('kaggle.json', kaggle_dir / 'kaggle.json')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('kaggle.json placed at', kaggle_dir / 'kaggle.json')

## 6 — Download ImageNet validation set (~6 GB)
This downloads only the val split — not the full 150 GB dataset.

In [ ]:
# Upgrade kaggle to latest and list competition files
!pip install -q --upgrade kaggle
import kaggle
kaggle.api.authenticate()

files = kaggle.api.competition_list_files('imagenet-object-localization-challenge')
for f in files:
    print(f.name, f.size)

## 6b — Download val split\nLook at the file listing above, find the validation tar filename, and paste it after `-f` below.

import glob, os

# Auto-detect whatever was downloaded (tar or zip)
tars = glob.glob('data/imagenet/*.tar') + glob.glob('data/imagenet/*.zip')
assert tars, 'No archive found in data/imagenet/ — check Cell 6b downloaded correctly'
archive = tars[0]
print(f'Extracting {archive} ...')

os.makedirs('data/imagenet/val', exist_ok=True)
if archive.endswith('.zip'):
    !unzip -q {archive} -d data/imagenet/val
else:
    !tar -xf {archive} -C data/imagenet/val

!echo "Extracted — $(ls data/imagenet/val | wc -l) files"

## 7 — Extract val images

In [ ]:
!mkdir -p data/imagenet/val
!tar -xf data/imagenet/val.tar -C data/imagenet/val
!echo "Extracted — $(ls data/imagenet/val | wc -l) files"

## 8 — Restructure into ImageFolder format
Kaggle dumps all val images flat into one folder. This step organises them into one subfolder per class so `torchvision.ImageFolder` can read them.

In [ ]:
!wget -q https://raw.githubusercontent.com/soumith/imagenetloader.torch/master/valprep.sh
%cd data/imagenet/val
!bash ../../../valprep.sh
%cd /content/ADVC-main
!echo "Done — $(ls data/imagenet/val | wc -l) class folders"

## 9 — Verify ImageFolder

In [ ]:
from torchvision.datasets import ImageFolder

ds = ImageFolder('data/imagenet/val')
print(f'Images  : {len(ds)}')
print(f'Classes : {len(ds.classes)}')
print(f'Sample  : {ds.classes[:5]} …')

## 10 — Run Phase 1 evaluation
Loops over `fp32 → int8 → int4` × `fgsm / pgd / patch`.  
Results are saved to `results/phase1_results.csv` after each combination.  
Safe to interrupt and re-run — completed rows are skipped automatically.

In [ ]:
!python experiments/eval_phase1.py --model deit_small

## 11 — View results

In [ ]:
import pandas as pd

df = pd.read_csv('results/phase1_results.csv')
df['clean_acc']       = df['clean_acc'].map('{:.4f}'.format)
df['robust_acc']      = df['robust_acc'].map('{:.4f}'.format)
df['asr']             = df['asr'].map('{:.4f}'.format)
df['robustness_gap']  = df['robustness_gap'].map('{:.4f}'.format)
df

## 12 — (Optional) Save results to Google Drive
Colab wipes `/content` on runtime reset. Mount Drive to persist results.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/ADVC/results'
shutil.copytree('results', dest, dirs_exist_ok=True)
print(f'Results saved to {dest}')